# 1. Setup

In [0]:
# Install if needed 
#!pip install pyspark

from pyspark.sql import SparkSession

# Start Spark
spark = SparkSession.builder.appName("AdvancedAnalytics").getOrCreate()

# 2. Load Data

In [0]:

user_profile = spark.sql("select *from brightlearn.brighttv.user_profile")

user_profile.show()

In [0]:
viewership = spark.sql("select * from brightlearn.brighttv.viewership")

viewership.display()

# 3. Join

In [0]:
df_spark = viewership.join(user_profile, on ="UserID", how ="left")
df_spark.show(5)

In [0]:
df_spark.printSchema()

# Feature Engineering

In [0]:
from pyspark.sql import functions as  F

df_spark = df_spark.withColumn(
  "view_time",
  (F.hour("Duration 2") *3600 +
   F.minute("Duration 2") *60 +
   F.second("Duration 2"))/60)

# Aggregations

In [0]:
from pyspark.sql.functions import sum, avg, count

agg_df = df_spark.groupBy("Gender").agg(
    sum("view_time").alias("total_watch"),
    avg("view_time").alias("avg_watch"),
    count("UserID").alias("user_count")
)

agg_df.show()

In [0]:
spark.sql("select count(UserID) from brightlearn.brighttv.viewership").show()

In [0]:
%sql

select userid, view_time
from df_spark
where view_time > 30 --#1
order by view_time desc --#2

In [0]:
df_spark.filter(df_spark.view_time > 30)\
        .orderBy(df_spark.view_time.desc())\
        .select("UserID", "view_time")\
        .show(5)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("Gender").orderBy(df_spark.view_time.desc())

ranked = df_spark.withColumn("rank", row_number().over(window_spec))

ranked.display()